## M3 actualizado — Índice F-of-M completo
Fórmula completa con los 5 componentes originales del equipo:
F-of-M = 0.30·Solvencia + 0.25·Demanda + 0.25·Soledad + 0.15·Concentración + 0.05·Futuro

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('dataset_maestro.csv',
                 dtype={'seccion_censal': str},
                 encoding='utf-8-sig')
df_muni = pd.read_csv('municipios_limpios.csv')

print(f"Secciones: {df.shape[0]:,}")
print(f"Columnas:  {df.shape[1]}")
print(f"\nVariables disponibles:\n{df.columns.tolist()}")

Secciones: 36,334
Columnas:  24

Variables disponibles:
['seccion_censal', 'CPRO', 'CMUN', 'latitud', 'longitud', 'total_ambos', 'vol_65plus', 'pct_65plus', 'vol_75plus', 'pct_75plus', 'cohorte_futura', 'mujeres_80plus', 'hombres_80plus', 'ratio_viudez', 'renta_persona_2024', 'renta_hogar_2024', 'solvente', 'cohorte_futura_2029', 'CODAUTO', 'indice_cobertura', 'plazas_total', 'hogares_uni_70_2029', 'hosp_en_provincia', 'tiene_hospital']


### Paso 1 — Normalización min-max de todas las variables

In [2]:
def normalizar(serie):
    minimo = serie.min()
    maximo = serie.max()
    return (serie - minimo) / (maximo - minimo)

def normalizar_inverso(serie):
    minimo = serie.min()
    maximo = serie.max()
    return 1 - (serie - minimo) / (maximo - minimo)

# Componente Solvencia
df['n_renta'] = normalizar(df['renta_hogar_2024'])

# Componente Demanda
df['n_vol_75plus']  = normalizar(df['vol_75plus'])
df['n_pct_75plus']  = normalizar(df['pct_75plus'])

# Componente Soledad — ahora con hogares unipersonales también
df['n_ratio_viudez']     = normalizar(df['ratio_viudez'])
df['n_hogares_uni']      = normalizar(df['hogares_uni_70_2029'])

# Componente Concentración — inverso porque más competencia = peor
df['n_cobertura_inv']    = normalizar_inverso(df['indice_cobertura'])
df['n_hospital']         = normalizar(df['tiene_hospital'])

# Componente Futuro
df['n_cohorte_2029']     = normalizar(df['cohorte_futura_2029'])

cols_n = ['n_renta', 'n_vol_75plus', 'n_pct_75plus',
          'n_ratio_viudez', 'n_hogares_uni', 
          'n_cobertura_inv', 'n_hospital', 'n_cohorte_2029']

print("Variables normalizadas (todas deben estar entre 0 y 1):")
print(df[cols_n].agg(['min','max','mean']).round(3).to_string())

Variables normalizadas (todas deben estar entre 0 y 1):
      n_renta  n_vol_75plus  n_pct_75plus  n_ratio_viudez  n_hogares_uni  n_cobertura_inv  n_hospital  n_cohorte_2029
min     0.000         0.000         0.000           0.000          0.000            0.000        0.00           0.000
max     1.000         1.000         1.000           1.000          1.000            1.000        1.00           1.000
mean    0.258         0.051         0.191           0.071          0.372            0.599        0.51           0.042


### Paso 2 — Construir el F-of-M con los 5 componentes originales

In [3]:
# Pesos originales del equipo
W_SOLVENCIA     = 0.30
W_DEMANDA       = 0.25
W_SOLEDAD       = 0.25
W_CONCENTRACION = 0.15
W_FUTURO        = 0.05

# Componentes internos
comp_solvencia     = df['n_renta']
comp_demanda       = (df['n_vol_75plus'] + df['n_pct_75plus']) / 2
comp_soledad       = (df['n_ratio_viudez'] + df['n_hogares_uni'].fillna(0)) / 2
comp_concentracion = (df['n_cobertura_inv'] + df['n_hospital']) / 2
comp_futuro        = df['n_cohorte_2029']

# Fórmula completa
df['fofm'] = (
    W_SOLVENCIA     * comp_solvencia.fillna(0) +
    W_DEMANDA       * comp_demanda +
    W_SOLEDAD       * comp_soledad +
    W_CONCENTRACION * comp_concentracion +
    W_FUTURO        * comp_futuro
)

# Filtro duro — secciones no solventes = 0
df.loc[df['solvente'] == 0, 'fofm'] = 0

print("=== DISTRIBUCIÓN DEL F-of-M COMPLETO ===")
print(f"Secciones puntuadas (fofm > 0): {(df['fofm']>0).sum():,}")
print(f"Secciones descartadas:          {(df['fofm']==0).sum():,}")
print(f"\nMáximo:  {df['fofm'].max():.4f}")
print(f"Media:   {df[df['fofm']>0]['fofm'].mean():.4f}")
print(f"Mínimo (>0): {df[df['fofm']>0]['fofm'].min():.4f}")

df[['seccion_censal','renta_hogar_2024','vol_75plus',
    'ratio_viudez','indice_cobertura','fofm']].sort_values(
    'fofm', ascending=False).head(10)

=== DISTRIBUCIÓN DEL F-of-M COMPLETO ===
Secciones puntuadas (fofm > 0): 24,438
Secciones descartadas:          11,896

Máximo:  0.6048
Media:   0.2803
Mínimo (>0): 0.0770


,seccion_censal,renta_hogar_2024,vol_75plus,ratio_viudez,indice_cobertura,fofm
20622,2811501002,83812.0,229.0,4.416667,2.943155,0.604764
20621,2811501001,83812.0,250.0,3.794118,2.943155,0.599585
20659,2811502009,83812.0,217.0,2.200000,2.943155,0.596686
20651,2811502001,83812.0,183.0,3.000000,2.943155,0.595072
20652,2811502002,83812.0,222.0,3.027027,2.943155,0.593996
20624,2811501005,83812.0,141.0,2.062500,2.943155,0.593016
20627,2811501008,83812.0,179.0,1.313725,2.943155,0.592838
20626,2811501007,83812.0,123.0,2.120000,2.943155,0.591851
20628,2811501009,83812.0,142.0,1.718750,2.943155,0.591177
20641,2811501022,83812.0,227.0,0.924528,2.943155,0.590486


### Paso 3 — Agregar por municipio y seleccionar top 100

In [4]:
# Añadir nombre de municipio
df_muni['CPRO'] = df_muni['CPRO'].astype(int)
df_muni['CMUN'] = df_muni['CMUN'].astype(int)
df['CPRO'] = df['CPRO'].astype(float).astype('Int64')
df['CMUN'] = df['CMUN'].astype(float).astype('Int64')

df = df.merge(df_muni[['CPRO','CMUN','NOMBRE']], on=['CPRO','CMUN'], how='left')

# Agregar por municipio
df_municipios = df[df['fofm'] > 0].groupby(['CPRO','CMUN','NOMBRE']).agg(
    secciones          = ('seccion_censal', 'count'),
    fofm_medio         = ('fofm', 'mean'),
    fofm_max           = ('fofm', 'max'),
    vol_75plus_total   = ('vol_75plus', 'sum'),
    pct_75plus_media   = ('pct_75plus', 'mean'),
    ratio_viudez_media = ('ratio_viudez', 'mean'),
    cohorte_futura     = ('cohorte_futura_2029', 'sum'),
    renta_hogar        = ('renta_hogar_2024', 'mean'),
    indice_cobertura   = ('indice_cobertura', 'first'),
    tiene_hospital     = ('tiene_hospital', 'max'),
    latitud            = ('latitud', 'mean'),
    longitud           = ('longitud', 'mean')
).reset_index()

df_municipios = df_municipios.sort_values('fofm_medio', ascending=False).reset_index(drop=True)
df_municipios['ranking_municipio'] = df_municipios.index + 1

# Filtro tamaño mínimo y diversificación geográfica
MINIMO_75PLUS = 500

df_candidatos = df_municipios[
    df_municipios['vol_75plus_total'] >= MINIMO_75PLUS
].copy()

top100 = (df_candidatos
    .groupby('CPRO')
    .apply(lambda x: x.nlargest(3, 'fofm_medio'))
    .reset_index(drop=True)
    .nlargest(100, 'fofm_medio')
    .reset_index(drop=True)
)
top100['posicion_final'] = top100.index + 1

print(f"Municipios candidatos: {len(df_candidatos):,}")
print(f"Ubicaciones seleccionadas: {len(top100)}")
print(f"Provincias representadas:  {top100['CPRO'].nunique()}")
print(f"\n=== TOP 20 ===")
print(top100[['posicion_final','NOMBRE','renta_hogar',
              'vol_75plus_total','indice_cobertura',
              'fofm_medio']].head(20).to_string(index=False))

Municipios candidatos: 700
Ubicaciones seleccionadas: 100
Provincias representadas:  49

=== TOP 20 ===
 posicion_final                 NOMBRE  renta_hogar  vol_75plus_total  indice_cobertura  fofm_medio
              1     Pozuelo de Alarcón      83812.0            7192.0          2.943155    0.581482
              2     Boadilla del Monte      65201.0            3011.0          2.943155    0.496628
              3            Majadahonda      60937.0            5384.0          2.943155    0.486806
              4  Sant Cugat del Vallès      58164.0            5967.0          2.912269    0.471096
              5             Matadepera      68286.0             757.0          2.912269    0.443240
              6 Esplugues de Llobregat      38952.0            5434.0          2.912269    0.406877
              7               Valencia      30725.0           84138.0          1.768381    0.323539
              8 Donostia/San Sebastián      41167.0           23273.0          2.773233    0.317

### Paso 4 — Guardar resultados actualizados

In [5]:
# Guardar archivos
df_municipios.to_csv('ranking_municipios.csv', index=False, encoding='utf-8-sig')
top100.to_csv('top100_ubicaciones.csv', index=False, encoding='utf-8-sig')

resumen_provincia = (top100.groupby('CPRO').agg(
    municipios   = ('NOMBRE', 'count'),
    fofm_medio   = ('fofm_medio', 'mean'),
    vol_75plus   = ('vol_75plus_total', 'sum'),
    renta_media  = ('renta_hogar', 'mean')
).sort_values('fofm_medio', ascending=False).reset_index())
resumen_provincia.to_csv('resumen_provincias.csv', index=False, encoding='utf-8-sig')

# Excel actualizado
df_muni_load = pd.read_csv('municipios_limpios.csv')
with pd.ExcelWriter('resultados_akademia.xlsx', engine='openpyxl') as writer:
    top100.to_excel(writer, sheet_name='Top 100 ubicaciones', index=False)
    df_municipios.head(500).to_excel(writer, sheet_name='Ranking municipios', index=False)
    resumen_provincia.to_excel(writer, sheet_name='Resumen por provincia', index=False)

print("Archivos guardados:")
print(f"  ranking_municipios.csv → {len(df_municipios):,} municipios")
print(f"  top100_ubicaciones.csv → {len(top100)} ubicaciones")
print(f"  resumen_provincias.csv → {resumen_provincia.shape[0]} provincias")
print(f"  resultados_akademia.xlsx → actualizado")

Archivos guardados:
  ranking_municipios.csv → 2,825 municipios
  top100_ubicaciones.csv → 100 ubicaciones
  resumen_provincias.csv → 49 provincias
  resultados_akademia.xlsx → actualizado


In [6]:
# 1. Los pesos suman 1.0
pesos = W_SOLVENCIA + W_DEMANDA + W_SOLEDAD + W_CONCENTRACION + W_FUTURO
print(f"Suma de pesos: {pesos} (debe ser 1.0)")

# 2. fofm siempre entre 0 y 1
print(f"\nfofm fuera de rango [0,1]: {((df['fofm']<0) | (df['fofm']>1)).sum()}")

# 3. Secciones no solventes tienen fofm=0
no_solventes_con_fofm = df[(df['solvente']==0) & (df['fofm']>0)]
print(f"Secciones no solventes con fofm>0: {len(no_solventes_con_fofm)} (debe ser 0)")

# 4. El índice de cobertura penaliza correctamente
# Castilla y León (CODAUTO=7, más saturada) debe puntuar menos que
# Comunitat Valenciana (CODAUTO=10, menos saturada) en n_cobertura_inv
cyl = df[df['CODAUTO']==7]['n_cobertura_inv'].mean()
val = df[df['CODAUTO']==10]['n_cobertura_inv'].mean()
print(f"\nn_cobertura_inv Castilla y León: {cyl:.3f} (debe ser bajo)")
print(f"n_cobertura_inv C. Valenciana:   {val:.3f} (debe ser alto)")

# 5. Top 3 tienen sentido — renta alta, demanda alta, cobertura baja
print("\n=== TOP 5 VERIFICACIÓN ===")
print(top100[['NOMBRE','renta_hogar','vol_75plus_total',
              'indice_cobertura','fofm_medio']].head(5).to_string(index=False))

# 6. Ningún municipio descartado por tamaño aparece en el top 100
min_vol = top100['vol_75plus_total'].min()
print(f"\nMínimo vol_75plus en top 100: {min_vol:.0f} (debe ser >= {MINIMO_75PLUS})")

# 7. Máximo 3 municipios por provincia
max_por_provincia = top100.groupby('CPRO').size().max()
print(f"Máximo municipios por provincia: {max_por_provincia} (debe ser <= 3)")

Suma de pesos: 1.0 (debe ser 1.0)

fofm fuera de rango [0,1]: 0
Secciones no solventes con fofm>0: 0 (debe ser 0)

n_cobertura_inv Castilla y León: 0.000 (debe ser bajo)
n_cobertura_inv C. Valenciana:   0.838 (debe ser alto)

=== TOP 5 VERIFICACIÓN ===
               NOMBRE  renta_hogar  vol_75plus_total  indice_cobertura  fofm_medio
   Pozuelo de Alarcón      83812.0            7192.0          2.943155    0.581482
   Boadilla del Monte      65201.0            3011.0          2.943155    0.496628
          Majadahonda      60937.0            5384.0          2.943155    0.486806
Sant Cugat del Vallès      58164.0            5967.0          2.912269    0.471096
           Matadepera      68286.0             757.0          2.912269    0.443240

Mínimo vol_75plus en top 100: 502 (debe ser >= 500)
Máximo municipios por provincia: 3 (debe ser <= 3)
